In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [2]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import pickle
import math
import scipy
from pathlib import Path
import sys
import os
import warnings
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
from matplotlib.ticker import ScalarFormatter
from copy import deepcopy
sys.path.append("../src")
from custom_distance import KL, conditionKL
import itertools
import pickle

from utils import number_split, create_mix, appendMetrics
from data_process import load_wls_adress_AddDomain
from process_SHAC import load_process_SHAC
from custom_distance import KL
from process_HateSpeech import load_HateSpeech_dynGen, load_HateSpeech_wsf
from process_CD import load_cd

import random
from tqdm import tqdm

warnings.filterwarnings("ignore")


from augmentation import reverseTranslationDF

In [3]:
# dataset_name = "CD"
# dataset_name = "HateSpeech"
dataset_name = "SHAC"

In [4]:

######## Load Data
if dataset_name == "SHAC":
    df_shac = load_process_SHAC(replaceNA="all")
    df_shac["label_binary"] = df_shac.apply(lambda x: 1 if x["Drug"] else 0, axis=1)
    df_shac["dfSource"] = df_shac["location"]

    df_shac_uw = df_shac.query("location == 'uw'").reset_index(drop=True)
    df_shac_mimic = df_shac.query("location == 'mimic'").reset_index(drop=True)

    n_test = 200
    
    z_Categories = ["uw", "mimic"]  # the order here matters! Should match with df0, df1
    label = "Drug"
    n_zCats = len(z_Categories)
    txt_col = "text"
    domain_col = "location"
    df0 = df_shac_uw
    df1 = df_shac_mimic
elif dataset_name == "HateSpeech":
    df_dynGen = load_HateSpeech_dynGen()
    df_wsf = load_HateSpeech_wsf()

    # n_test = 1000
    n_test = 200
    
    z_Categories = [
        "dynGen",
        "wsf",
    ]  # the order here matters! Should match with df0, df1
    label = "label_binary"
    n_zCats = len(z_Categories)
    txt_col = "text"
    domain_col = "dfSource"
    df0 = df_dynGen
    df1 = df_wsf
elif dataset_name == "CD":
    df_all = load_cd()
    df_avh = df_all["avh"]
    df_r56 = df_all["r56"]

    n_test = 200
    
    z_Categories = ["avh", "r56"]
    label = "label_binary"
    n_zCats = len(z_Categories)
    txt_col = "text"
    domain_col = "dfSource"
    df0 = df_avh
    df1 = df_r56


In [5]:
df0['ssid'] = ["df0_" + str(x) for x in np.arange(len(df0))]
df1['ssid'] = ["df1_" + str(x) for x in np.arange(len(df1))]

# Old MT

In [6]:
src = 'en'
tgt = 'de'

In [7]:
outdir = f"../output/ReverseTranslate/{dataset_name}"
os.makedirs(outdir, exist_ok=True)

In [ ]:
tmp = reverseTranslationDF(df_in=df0, src=src, tgt=tgt, txt_col=txt_col, save=True, outfile=f"{outdir}/df0_{tgt}.csv", device="cuda:0", max_length=512)
tmp = reverseTranslationDF(df_in=df1, src=src, tgt=tgt, txt_col=txt_col, save=True, outfile=f"{outdir}/df1_{tgt}.csv", device="cuda:0", max_length=512)

In [18]:
len(df0)

5424

In [9]:
for i in range(10):
    print(tmp['text'].iloc[i])
    print(tmp['Text'].iloc[i])
    print("")

I'm busy with my infusion, which means I have to go to the hospital.
I'm busy with my infusion, which means I have to go to the hospital.

No hearing voices then.
Then we don't hear voices.

When I got home and I was alone in my room, I heard my name.
When I came home and was alone in my room, I heard my name.

Weirdly enough, the voice was male this time.
Strangely enough, this time the voice was male.

Okay.
All right.

<DATE_TIME> seems like it's going to be a pretty good day.
<DATE_TIME> seems to be a pretty good day.

I had a real good sleep.
I had a good night's sleep.

The only thing that woke me up was the voices in my head saying it's time to get up.
The only thing that woke me up was the voices in my head saying it's time to get up.

I don't know.
I don't know.

Sometimes they have good comments, most of the time they don't.
Sometimes they have good comments, mostly not.



In [20]:
reverseTranslationDF(df_in=df0.iloc[:5], src=src, tgt=tgt, txt_col=txt_col, save=False, outfile=None, device="cuda:0")['Text'].values

array(["I'm busy with my infusion, which means I have to go to the hospital.",
       "Then we don't hear voices.",
       'When I came home and was alone in my room, I heard my name.',
       'Strangely enough, this time the voice was male.', 'All right.'],
      dtype=object)

In [21]:
reverseTranslationDF(df_in=df0.iloc[:5], src=src, tgt=tgt, txt_col=txt_col, save=False, outfile=None, device="cuda:0")['Text'].values

array(["I'm busy with my infusion, which means I have to go to the hospital.",
       "Then we don't hear voices.",
       'When I came home and was alone in my room, I heard my name.',
       'Strangely enough, this time the voice was male.', 'All right.'],
      dtype=object)

In [22]:
reverseTranslationDF(df_in=df0.iloc[:5], src=src, tgt=tgt, txt_col=txt_col, save=False, outfile=None, device="cuda:0")['Text'].values

array(["I'm busy with my infusion, which means I have to go to the hospital.",
       "Then we don't hear voices.",
       'When I came home and was alone in my room, I heard my name.',
       'Strangely enough, this time the voice was male.', 'All right.'],
      dtype=object)

# SeamlessM4T-v2

In [11]:
processor = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
model = SeamlessM4Tv2Model.from_pretrained("facebook/seamless-m4t-v2-large").to("cuda")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [31]:
def reverseTranslateSeamlessM4T(df_in, txt_col, src="eng", tgt="fra", device="cuda", save=False, outfile=None):
    processor = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
    model = SeamlessM4Tv2Model.from_pretrained("facebook/seamless-m4t-v2-large").to(device)
    
    
    df = deepcopy(df_in)
    
    translated_text_reverse = []
    
    for i in tqdm(range(len(df)), file=open("../log/reverseTranslate.txt", "w")):
        text_original = df[txt_col].iloc[i]

        text_inputs = processor(text = text_original, src_lang=src, return_tensors="pt").to(device)


        # from Original to Target Language
        output_tokens = model.generate(**text_inputs, tgt_lang=tgt, generate_speech=False)
        translated_text_from_text = processor.decode(output_tokens[0].tolist()[0], skip_special_tokens=True)

        text_inputs_for_reverse = processor(text = translated_text_from_text, src_lang=tgt, return_tensors="pt").to(device)


        # from Target Language to Original Language (Reverse Translate)
        output_tokens = model.generate(**text_inputs_for_reverse, tgt_lang=src, generate_speech=False)
        text_output = processor.decode(output_tokens[0].tolist()[0], skip_special_tokens=True)

        translated_text_reverse.append(text_output)
        
    df['reverseTranslateText'] = translated_text_reverse
    
    if save:
        df.to_csv(outfile, index=False)

    return df

In [32]:
outdir = f"../output/ReverseTranslate/{dataset_name}"
os.makedirs(outdir, exist_ok=True)

In [33]:
tmp = reverseTranslateSeamlessM4T(df_in = df0, txt_col=txt_col, src="eng",tgt="fra", device="cuda", save=True, outfile=f"{outdir}/df0.csv")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


KeyboardInterrupt



In [2]:
import pandas as pd

In [3]:
df0_aug = pd.read_csv(f"../output/ReverseTranslate/CD/df0_de.csv")

In [5]:
txt_col = 'text'

In [6]:
df0_aug.drop(txt_col, axis=1, inplace=True)

In [14]:
df0_aug.rename(columns={"text_translated_reverse": txt_col})

,Unnamed: 0,study_id,annotator_id,avh_id,sent_id,ND,DT,L,O,MF,...,AR,AI,message_fold,client_fold,label_binary,label,dfSource,ssid,text_translated_forward,text
0,0,1,0,u00001966@avh-20190731-1,S0,1,0.0,0.0,0.0,0.0,...,0.0,0.0,1,2,0,0,avh,df0_0,"Ich bin mit meiner Infusion beschäftigt, was b...","I'm busy with my infusion, which means I have ..."
1,1,1,0,u00001966@avh-20190731-1,S1,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0,2,0,0,avh,df0_1,Dann hören wir keine Stimmen.,Then we don't hear voices.
2,2,1,0,u00001966@avh-20190731-1,S2,1,0.0,0.0,0.0,0.0,...,0.0,0.0,3,2,0,0,avh,df0_2,Als ich nach Hause kam und allein in meinem Zi...,"When I came home and was alone in my room, I h..."
3,3,1,0,u00001966@avh-20190731-1,S3,1,0.0,0.0,0.0,0.0,...,0.0,0.0,1,2,0,0,avh,df0_3,Seltsamerweise war die Stimme dieses Mal männl...,"Strangely enough, this time the voice was male."
4,4,2,0,u00001035@avh-20181105-1,S0,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0,2,0,0,avh,df0_4,In Ordnung.,All right.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5419,5419,510,2,u00001706@avh-20190325-1,S5,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0,3,0,0,avh,df0_5419,"Ähm, nicht so, wie ich es gerne hätte.","Um, not the way I'd like it to be."
5420,5420,510,2,u00001706@avh-20190325-1,S6,0,0.0,0.0,1.0,0.0,...,0.0,0.0,2,3,1,1,avh,df0_5420,Es funktioniert einfach nicht.,It just doesn't work.
5421,5421,511,2,u00002028@avh-20190719-1,S0,0,0.0,0.0,0.0,0.0,...,0.0,1.0,0,2,1,1,avh,df0_5421,"Also, meine Stimmen - es klingt so verrückt, w...","So, my voices -- it sounds so crazy when I say..."
5422,5422,511,2,u00002028@avh-20190719-1,S1,1,0.0,0.0,0.0,0.0,...,0.0,0.0,0,2,0,0,avh,df0_5422,"Aber wie, es ist vor allem, wie, ein voll ausg...","But like, it's above all, like, a full blown c..."
